# MLP Model Multivariate

In this section we implement multivariate forecasting using the MLP Model with the **TimeSeriesDatasetVectorizedExog** approach.

The MLP (Multi-Layer Perceptron) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **fTimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

The model architecture remains unchanged - we simply reshape the data to process all series in parallel, achieving 585x faster training while incorporating exogenous variables.


**Layer Breakdown:**

- **Input Flattening**: Reshapes (batch_size, seq_length, input_size) → (batch_size, seq_length × input_size)
- **Hidden Layers**: 3 stacked fully connected layers with ReLU activation
- **Hidden Size**: 512 units per layer
- **Dropout**: Applied after each hidden layer (0.2)
- **Output Layer**: Single fully connected layer producing 1-step forecast

In [1]:
import torch
import torch.nn as nn

## Model

In [ ]:
class MLPForecaster(nn.Module):
    """
    MLP model for UNIVARIATE time series forecasting with one-hot encoding.
    Architecture: 
        Input Flattening -> MLP Layers (Linear -> ReLU -> Dropout) -> Output
    
    Takes input sequences with multiple features (Value + features + year + month + one-hot)
    and flattens them before processing through fully connected layers.
    
    Unlike MLPMultivariate:
    - Uses one-hot encoding to identify individual time series
    - Predicts one value at a time for a specific series
    - Processes entire sequence as flattened vector
    
    Good for:
    - Capturing non-linear patterns across the entire sequence
    - Faster training than RNN/LSTM for shorter sequences
    - Learning complex feature interactions
    """
    def __init__(self, input_size, seq_length, hidden_size=512, num_layers=3, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            seq_length: Sequence length (lookback window)
            hidden_size: Number of units in each MLP layer
            num_layers: Number of MLP layers
            dropout: Dropout rate
        """
        super(MLPForecaster, self).__init__()
        
        self.input_size = input_size
        self.seq_length = seq_length
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Calculate input dimension after flattening
        self.input_dim = seq_length * input_size
        
        # Build MLP layers
        layers = []
        
        # First layer
        layers.append(nn.Linear(self.input_dim, hidden_size))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        
        self.mlp = nn.Sequential(*layers)
        
        # Output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        """
        Args:
            x: Input tensor of shape (batch_size, seq_length, input_size)
        
        Returns:
            predictions: (batch_size, 1) - single value prediction
        """
        batch_size = x.size(0)
        
        # Flatten entire sequence
        x_flat = x.reshape(batch_size, -1)  # (batch_size, seq_length * input_size)
        
        # Pass through MLP
        x = self.mlp(x_flat)  # (batch_size, hidden_size)
        
        # Output prediction
        out = self.fc(x)  # (batch_size, 1)
        
        return out


### Model Results without Exogenous Features

### Model Results with Exogenous Features
